# Supplementary Results 8 — Drug target enrichment bias from sample size and therapeutic area

Does the 3.62-fold enrichment of approved target-indication pairs survive controlling for the
size of the GWAS behind each disease and for the therapeutic area it belongs to? Four models over
the full ChEMBL pair table: genetic support alone, plus scaled maximum sample size, plus a random
therapeutic-area intercept, and both together.

Numbers are written to `results/sr08_enrichment_bias.json`.

**Provenance.** `~/Projects/EGL_and_training_set/archive/gentropy_paper/R_scripts/05_enrichment.R`,
which fitted the two fixed-effect models with `glm` and the two mixed models with `lme4::glmer`,
on a table it called `data_for_drug_enrichment_ta.csv` — pairs annotated with `maxNSamples` and
`mappedTherapeuticAreas`, restricted to pairs whose disease maps to a therapeutic area. That
table is rebuilt below from `df_for_enrichment_regression.csv`.

**The mixed models are fitted in R, by `08_enrichment_bias.R` in this directory**, which this
notebook writes its model frame for and then calls. `lme4` was added to the project R library for
it (`chapters/r-env`), so `glmer` is the same estimator the published analysis used rather than an
approximation of it; statsmodels' variational `BinomialBayesMixedGLM` is still fitted alongside, to
show what the approximation costs.

In [1]:
import os
import subprocess

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

from manuscript_methods import paper

numbers = {}
MODEL_FRAME = paper.derived("sr8_model_frame.csv")
MODEL_RESULTS = paper.derived("sr8_enrichment_models.csv")
R_SCRIPT = "chapters/03-analysis-supplementary/08_enrichment_bias.R"

## The pair table, annotated with sample size and therapeutic area

The maximum sample size behind a disease is the largest `nSamples` of any GWAS study mapped to it;
the therapeutic area is the disease's primary area, on the hierarchy the drug analyses use.

In [2]:
pairs = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))
print(f"target-indication pairs: {len(pairs):,} | approved: {int(pairs['outcome'].sum()):,}")

studies = pd.read_parquet(paper.derived("study_annotation"), columns=["studyId", "nSamples", "diseaseIds"])
per_disease = (
    studies.explode("diseaseIds")
    .dropna(subset=["diseaseIds"])
    .groupby("diseaseIds")["nSamples"]
    .max()
    .rename("maxNSamples")
    .reset_index()
    .rename(columns={"diseaseIds": "diseaseId"})
)

areas = pd.read_parquet(paper.derived("efo_therapeutic_area")).rename(
    columns={"id": "diseaseId", "primaryTherapeuticArea": "therapeuticArea"}
)[["diseaseId", "therapeuticArea"]]

annotated = pairs.merge(per_disease, on="diseaseId", how="left").merge(areas, on="diseaseId", how="left")
annotated["maxNSamples"] = annotated["maxNSamples"].fillna(0)
print(annotated["therapeuticArea"].isna().sum(), "pairs have no therapeutic area")
print(f"pairs with a therapeutic area: {annotated['therapeuticArea'].notna().sum():,}")

target-indication pairs: 37,377 | approved: 4,564
7309 pairs have no therapeutic area
pairs with a therapeutic area: 30,068


In [3]:
# The R script dropped pairs whose disease maps to no therapeutic area.
model_frame = annotated[annotated["therapeuticArea"].notna()].copy()
model_frame["maxNSamplesScaled"] = (model_frame["maxNSamples"] - model_frame["maxNSamples"].mean()) / model_frame[
    "maxNSamples"
].std()
print(f"pairs in the models: {len(model_frame):,} | approved: {int(model_frame['outcome'].sum()):,}")
print(f"therapeutic areas: {model_frame['therapeuticArea'].nunique()}")

pairs in the models: 30,068 | approved: 3,914
therapeutic areas: 23


## Fixed-effect models

In [4]:
def logistic(frame, covariates, label):
    """Logistic regression of approval on genetic support and the given covariates."""
    design = sm.add_constant(frame[["geneticSupport", *covariates]])
    fitted = sm.Logit(frame["outcome"], design).fit(disp=False)
    return {
        "model": label,
        "OR for genetic support": round(float(np.exp(fitted.params["geneticSupport"])), 2),
        "beta": round(float(fitted.params["geneticSupport"]), 4),
        "P": float(fitted.pvalues["geneticSupport"]),
    }


# The published baseline of 3.62 is the main text's, over every pair; the covariate models run on
# the pairs whose disease maps to a therapeutic area, which on its own moves the estimate to 3.48.
fixed = pd.DataFrame(
    [
        logistic(annotated, [], "genetic support alone, every pair"),
        logistic(model_frame, [], "genetic support alone, pairs with a therapeutic area"),
        logistic(model_frame, ["maxNSamplesScaled"], "with maximum sample size"),
    ]
)
numbers["S8.01"] = fixed.loc[0, "OR for genetic support"]
numbers["S8.02"] = fixed.loc[2, "OR for genetic support"]
fixed

,model,OR for genetic support,beta,P
0,"genetic support alone, every pair",3.62,1.2861,3.364915e-58
1,"genetic support alone, pairs with a therapeuti...",3.48,1.2468,1.568173e-53
2,with maximum sample size,3.44,1.2355,2.900178e-52


## Random therapeutic-area intercept

Fitted by `08_enrichment_bias.R` with `lme4::glmer`, the estimator the published analysis used. The
two fixed-effect models are refitted there as well, so the R output can be read on its own.

In [5]:
# `glmer` is fitted in R: the notebook hands over the model frame and reads the results back.
model_frame.to_csv(MODEL_FRAME, index=False)
run = subprocess.run(
    ["bash", "tools/run_r.sh", R_SCRIPT],
    cwd=paper.ROOT,
    capture_output=True,
    text=True,
    env={**os.environ, "PATH": f"/opt/homebrew/bin:{os.environ.get('PATH', '')}"},
)
print(run.stdout.strip())
if run.returncode:
    raise RuntimeError(f"{R_SCRIPT} failed ({run.returncode}):\n{run.stderr}")

glmer = pd.read_csv(MODEL_RESULTS).set_index("model")
numbers["S8.03"] = round(float(glmer.loc["random therapeutic area only", "or"]), 2)
numbers["S8.04"] = round(float(glmer.loc["random therapeutic area and sample size", "or"]), 2)
numbers["S8.05"] = round(float(glmer.loc["random therapeutic area and sample size", "taVariance"]), 2)
numbers["S8.06"] = round(float(glmer.loc["random therapeutic area and sample size", "taSd"]), 2)
# Results 6 quotes the fully adjusted odds ratio; this is the only place it is computed.
numbers["R6.04"] = numbers["S8.04"]
glmer.round(4)

pairs 30068 | approved 3914 | therapeutic areas 23
                                   model     beta       or taVariance      taSd
                   genetic support alone 1.246815 3.479243         NA        NA
                with maximum sample size 1.235491 3.440066         NA        NA
            random therapeutic area only 1.168353 3.216690  0.5203495 0.7213525
 random therapeutic area and sample size 1.144870 3.142033  0.5259906 0.7252521
written: /Users/yt4/Projects/Gentropy-manuscript/data/intermediate_files_refactor/sr8_enrichment_models.csv


,beta,or,taVariance,taSd
model,,,,
genetic support alone,1.2468,3.4792,NaN,NaN
with maximum sample size,1.2355,3.4401,NaN,NaN
random therapeutic area only,1.1684,3.2167,0.5203,0.7214
random therapeutic area and sample size,1.1449,3.1420,0.5260,0.7253


### What the variational approximation costs

Before `lme4` was available these two models were fitted with statsmodels'
`BinomialBayesMixedGLM`, a variational approximation to the same likelihood. It is kept here for
comparison: it moves the fully adjusted odds ratio by 0.01 and the therapeutic-area variance by
0.03, which is why the published values could not be reproduced from it.

In [6]:
def variational(frame, covariates, label):
    """The same model through statsmodels\' variational approximation."""
    formula = " + ".join(["outcome ~ geneticSupport", *covariates]) if covariates else "outcome ~ geneticSupport"
    fitted = BinomialBayesMixedGLM.from_formula(formula, {"therapeuticArea": "0 + C(therapeuticArea)"}, frame).fit_vb(
        verbose=False
    )
    index = list(fitted.model.exog_names).index("geneticSupport")
    variance = float(np.exp(2 * fitted.vcp_mean[0]))
    return {
        "model": label,
        "OR, variational": round(float(np.exp(fitted.fe_mean[index])), 2),
        "TA variance, variational": round(variance, 2),
        "OR, glmer": round(float(glmer.loc[label, "or"]), 2),
        "TA variance, glmer": round(float(glmer.loc[label, "taVariance"]), 2),
    }


pd.DataFrame(
    [
        variational(model_frame, [], "random therapeutic area only"),
        variational(model_frame, ["maxNSamplesScaled"], "random therapeutic area and sample size"),
    ]
)

,model,"OR, variational","TA variance, variational","OR, glmer","TA variance, glmer"
0,random therapeutic area only,3.21,0.55,3.22,0.52
1,random therapeutic area and sample size,3.13,0.56,3.14,0.53


## Write the results

In [7]:
print(paper.save_results("sr08_enrichment_bias", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr08_enrichment_bias.json


,computed
S8.01,3.62
S8.02,3.44
S8.03,3.22
S8.04,3.14
S8.05,0.53
S8.06,0.73
R6.04,3.14
